In [1]:
pip install gymnasium[atari] torch torchvision stable-baselines3 opencv-python numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━ 483.0/664.8 MB 55.6 MB/s eta 0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 11.5 MB/s eta 0:00:00
ERROR: THESE PACKAGES DO NOT MATCH THE HASHES FROM THE REQUIREMENTS FILE. If you have updated the package versions, pleas

In [3]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import cv2
from collections import deque

# Hyperparameters
GAMMA = 0.99  # Discount factor
LR = 1e-4  # Learning rate
BUFFER_SIZE = 100000  # Experience replay buffer size
BATCH_SIZE = 32  # Mini-batch size
EPSILON_START = 1.0  # Initial exploration rate
EPSILON_END = 0.05  # Final exploration rate
EPSILON_DECAY = 1000000  # Steps over which epsilon decays
TARGET_UPDATE = 10000  # Target network update frequency
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Preprocessing function
def preprocess(obs):
    obs = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)  # Convert to grayscale
    obs = cv2.resize(obs, (84, 84))  # Resize to 84x84
    return np.array(obs, dtype=np.uint8) / 255.0  # Normalize

# DQN Network
class DQN(nn.Module):
    def __init__(self, action_dim):
        super(DQN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
            nn.Linear(512, action_dim)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# Replay Buffer
class ReplayMemory:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (torch.tensor(states, dtype=torch.float32, device=DEVICE),
                torch.tensor(actions, dtype=torch.long, device=DEVICE),
                torch.tensor(rewards, dtype=torch.float32, device=DEVICE),
                torch.tensor(next_states, dtype=torch.float32, device=DEVICE),
                torch.tensor(dones, dtype=torch.float32, device=DEVICE))

    def __len__(self):
        return len(self.buffer)

# Agent with DQN
class Agent:
    def __init__(self, action_dim):
        self.policy_net = DQN(action_dim).to(DEVICE)
        self.target_net = DQN(action_dim).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())  # Sync networks
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=LR)
        self.memory = ReplayMemory(BUFFER_SIZE)
        self.steps_done = 0
        self.epsilon = EPSILON_START
        self.action_dim = action_dim

    def select_action(self, state):
        # Epsilon-greedy policy
        self.epsilon = EPSILON_END + (EPSILON_START - EPSILON_END) * np.exp(-1. * self.steps_done / EPSILON_DECAY)
        self.steps_done += 1
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        with torch.no_grad():
            return self.policy_net(torch.tensor(state, device=DEVICE, dtype=torch.float32).unsqueeze(0)).argmax().item()

    def optimize_model(self):
        if len(self.memory) < BATCH_SIZE:
            return
        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)

        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q_values = self.target_net(next_states).max(1)[0]
            expected_q_values = rewards + GAMMA * next_q_values * (1 - dones)

        loss = nn.functional.mse_loss(q_values, expected_q_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

# Train the agent
def train(env_name="PongNoFrameskip-v5", num_episodes=50000):
    env = gym.make(env_name, render_mode="rgb_array")
    action_dim = env.action_space.n
    agent = Agent(action_dim)
    total_rewards = []

    for episode in range(num_episodes):
        state, _ = env.reset()
        state = preprocess(state)
        state = np.stack([state] * 4, axis=0)  # Stacked frames
        total_reward = 0

        for t in range(10000):  # Max steps per episode
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            next_state = preprocess(next_state)
            next_state = np.concatenate((state[1:], np.expand_dims(next_state, 0)), axis=0)  # Update stacked frames

            agent.memory.push(state, action, reward, next_state, terminated)
            state = next_state
            total_reward += reward

            agent.optimize_model()

            if terminated or truncated:
                break

        if episode % 10 == 0:
            agent.update_target()

        total_rewards.append(total_reward)
        print(f"Episode {episode}, Reward: {total_reward}, Epsilon: {agent.epsilon:.4f}")

    env.close()
    torch.save(agent.policy_net.state_dict(), "dqn_atari.pth")

# Run training
if __name__ == "__main__":
    train()

NameNotFound: Environment `PongNoFrameskip` doesn't exist.